# Training on Kaggle — 2×T4 GPU

Complete guide to training the ACT or Diffusion policy on Kaggle free GPUs.

**Setup:** Kaggle → New Notebook → GPU T4 ×2 · Internet ON · 9h session

Add these secrets in _Kaggle → Add-ons → Secrets_ before running:
- `HF_TOKEN` — HuggingFace token with write access
- `WANDB_API_KEY` — (optional) Weights & Biases API key

In [ ]:
# Cell 1 — Install dependencies
# Kaggle already has torch; we install the remaining packages
!pip install lerobot==0.5.1 mujoco==3.5.0 mujoco-mjx==3.8.0 \
    mujoco_playground==0.2.0 hydra-core==1.3.2 mlflow==3.12.0 \
    loguru peft huggingface_hub -q

# Clone and install this repo
!git clone https://github.com/mefiezvous/lerobot-playground-portfolio.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -e . -q

In [ ]:
# Cell 2 — Configure secrets
import os
from kaggle_secrets import UserSecretsClient  # type: ignore[import]

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

try:
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("WandB enabled")
except Exception:
    print("WandB secret not found — MLflow only")

In [ ]:
# Cell 3 — Launch ACT training (Kaggle profile: 100k steps)
!python train.py --config-name training/kaggle policy=act \
    logging.run_name=kaggle_act_001 \
    training.device=cuda

In [ ]:
# Cell 4 — Resume from checkpoint (re-run this cell after session expiry)
# Auto-detection picks the latest checkpoint_XXXXXXXX.ckpt in checkpoints/
!python train.py --config-name training/kaggle policy=act \
    logging.run_name=kaggle_act_001_resume \
    training.device=cuda \
    training.checkpoint_dir=checkpoints/

In [ ]:
# Cell 5 — (Optional) Diffusion policy training
!python train.py --config-name training/kaggle policy=diffusion \
    logging.run_name=kaggle_diff_001 \
    training.device=cuda